In [1]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Input, Dense, GlobalAveragePooling2D, Lambda
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing import image
import tensorflow.keras.backend as K
from sklearn.model_selection import train_test_split

# Paths
base_path = "processed_signature/processed"
csv_path = os.path.join(base_path, "signature_dataset.csv")

df = pd.read_csv(csv_path)
print(df.head())

c:\Users\aviru\AppData\Local\Programs\Python\Python313\Lib\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/attr_value.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
c:\Users\aviru\AppData\Local\Programs\Python\Python313\Lib\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/tensor.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
c:\Users\aviru\AppData\Local\Programs\Python\Python313\Lib\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framewo

            image_path  person_id  label split
0  test/049/01_049.png         49      0  test
1  test/049/02_049.png         49      0  test
2  test/049/03_049.png         49      0  test
3  test/049/04_049.png         49      0  test
4  test/049/05_049.png         49      0  test


In [2]:
def load_image(img_path):
    # Load image as RGB
    img = image.load_img(os.path.join(base_path, img_path), target_size=(224,224))
    img = image.img_to_array(img)
    img = img / 255.0
    return img

In [3]:
print(df.columns)


Index(['image_path', 'person_id', 'label', 'split'], dtype='str')


In [4]:
def create_pairs(df):
    pairs = []
    labels = []

    grouped = df.groupby("person_id")
    for writer_id, group in grouped:
        genuine = group[group['label']==0]['image_path'].values
        forged  = group[group['label']==1]['image_path'].values

        # Genuine-Genuine pairs (label=1)
        for i in range(len(genuine)-1):
            img1 = load_image(genuine[i])
            img2 = load_image(genuine[i+1])
            pairs.append([img1,img2])
            labels.append(1)

        # Genuine-Forged pairs (label=0)
        for i in range(min(len(genuine),len(forged))):
            img1 = load_image(genuine[i])
            img2 = load_image(forged[i])
            pairs.append([img1,img2])
            labels.append(0)

    return np.array(pairs), np.array(labels)

pairs, labels = create_pairs(df)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(pairs, labels, test_size=0.2, random_state=42)

In [5]:
def l2_normalize(x):
    return K.l2_normalize(x, axis=1)

def build_embedding_model():
    base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224,224,3))
    base_model.trainable = True

    # Freeze early layers
    for layer in base_model.layers[:-40]:
        layer.trainable = False

    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = Dense(128, activation='relu')(x)
    x = Lambda(l2_normalize, output_shape=(128,))(x)  # ✅ specify output_shape

    return Model(base_model.input, x)

embedding_model = build_embedding_model()

In [6]:
def euclidean_distance(vectors):
    x, y = vectors
    return K.sqrt(K.sum(K.square(x - y), axis=1, keepdims=True))

input_a = Input(shape=(224,224,3))
input_b = Input(shape=(224,224,3))

embedding_a = embedding_model(input_a)
embedding_b = embedding_model(input_b)

distance = Lambda(euclidean_distance)([embedding_a, embedding_b])

siamese_model = Model(inputs=[input_a, input_b], outputs=distance)

# Contrastive loss
def contrastive_loss(y_true, y_pred):
    margin = 1
    return K.mean(
        y_true * K.square(y_pred) +
        (1 - y_true) * K.square(K.maximum(margin - y_pred, 0))
    )

siamese_model.compile(loss=contrastive_loss, optimizer=Adam(1e-4))
siamese_model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_2       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ functional          │ (None, 128)       │  2,421,952 │ input_layer_1[0]… │
│ (Functional)        │                   │            │ input_layer_2[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_1 (Lambda)   │ (None, 1)         │          0 │ functional[0][0], │
│                     │                   │            │ functional[1][0]  │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 2,421,952 (9.24 MB)

 Trainable params: 1,845,504 (7.04 MB)

 Non-trainable params: 576,448 (2.20 MB)

In [7]:
#Train Model
siamese_model.fit(
    [X_train[:,0], X_train[:,1]],
    y_train,
    validation_data=([X_test[:,0], X_test[:,1]], y_test),
    epochs=20,
    batch_size=8
)


KeyboardInterrupt: 

In [ ]:
embedding_model.save("embedding_model.keras")
siamese_model.save("signature_siamese_model.keras")

In [ ]:
customer_database = {}

def get_embedding(img_path):
    img = load_image(img_path)
    img = np.expand_dims(img, axis=0)
    return embedding_model.predict(img)

def register_customer(customer_id, folder_path):
    embeddings = []
    image_files = [
        os.path.join(folder_path, file)
        for file in os.listdir(folder_path)
        if file.lower().endswith(('.png','.jpg','.jpeg'))
    ]

    if not image_files:
        print("No images found in folder!")
        return

    for img_path in image_files:
        emb = get_embedding(img_path)
        embeddings.append(emb)

    customer_database[customer_id] = embeddings
    print(f"✅ Customer {customer_id} registered successfully. Stored {len(embeddings)} samples.")

In [ ]:
def verify_signature(customer_id, test_signature_path, threshold=0.6):
    if customer_id not in customer_database:
        print("Customer not found!")
        return

    test_emb = get_embedding(test_signature_path)
    stored_embeddings = customer_database[customer_id]

    distances = [np.linalg.norm(test_emb - emb) for emb in stored_embeddings]
    avg_distance = np.mean(distances)
    print("Average Distance:", avg_distance)

    if avg_distance < threshold:
        confidence = (1 - avg_distance/threshold) * 100
        print("✅ Genuine Signature")
        print("Confidence:", round(confidence,2), "%")
    else:
        confidence = (avg_distance/threshold - 1) * 100
        print("❌ Forged Signature")
        print("Confidence:", round(confidence,2), "%")

In [ ]:
#Balance Pairs
print("Positive pairs:", np.sum(labels==1))
print("Negative pairs:", np.sum(labels==0))

Positive pairs: 2085
Negative pairs: 1907


In [ ]:
# Register a customer
register_customer("C101", "processed")

# Verify a signature
verify_signature("C101", "more_forg.jpeg")

NameError: name 'register_customer' is not defined